# Chapter 3 — Select a coordinate and evaluate

Source candidate · CONVERGING · checkpoint-bound evidence

> **Source candidate / CONVERGING.** These cells describe the Chapter’s
> model and operations; their presence is not execution or scientific
> evidence. The website runs no kernels or solvers, and the generated
> Notebook remains zero-output. See the earlier diagram lessons for
> circuit review.

The same fixed resonator model now supports evaluation rather than a
full frequency response: what is a named quantity at its public retained
resonator coordinate? The Plan still owns wiring; the View selected
later is an independent analytical choice.

## Lesson 3.1 — Rebuild the fixed circuit

### Rebuild the physical declaration

The familiar sequence stays explicit: root and child ownership first,
fixed native parts, then local structure and boundary.

In [ ]:
from scnsim import CircuitPlan, components, units as u

plan = CircuitPlan(id="primitive_resonator")
resonator = plan.subsystem(id="resonator")

In [ ]:
capacitor = resonator.add(
    components.capacitor(id="capacitor", capacitance=110.0 * u.fF)
)
inductor = resonator.add(
    components.inductor(id="inductor", inductance=5.8 * u.nH)
)

The fixed leaves remain private; the parallel relation shares their
local terminal bus and the child ground.

In [ ]:
resonator_bus = resonator.bus(id="terminal")
parallel_lc = resonator.parallel(
    id="parallel_lc",
    start=resonator_bus,
    branches=((capacitor,), (inductor,)),
    end=resonator.ground,
)

In [ ]:
terminal = resonator.expose_pin(id="terminal", at=resonator_bus)

`terminal` is the single public electrical boundary for root wiring.

The root uses the public terminal, not either private LC leaf.

In [ ]:
signal_bus = plan.bus(id="signal_boundary")
resonator_root_bus = plan.bus(id="resonator_node")
coupling_cap = plan.add(
    components.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
)
coupling = plan.series(
    id="coupling",
    start=signal_bus,
    elements=(coupling_cap,),
    end=resonator_root_bus,
)
plan.link(
    id="resonator_terminal",
    endpoints=(resonator_root_bus, terminal),
)
signal_port = plan.add_port(
    id="signal_in",
    at=signal_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
resonator_node = resonator_root_bus.node

The root coordinate, rather than a private LC leaf, is ready for
selection.

## Lesson 3.2 — Ask for a root quantity

### Retain and evaluate the public coordinate

First derive the View that retains the root named-bus coordinate.

In [ ]:
from scnsim import CircuitRun, ReductionPipeline

run = CircuitRun(plan=plan, workspace="workspaces/primitive-course")
response_view = run.original
quantity_view = response_view.reduce(
    ReductionPipeline().retain(resonator_node)
)

`retain(resonator_node)` states the analytical coordinate boundary, and
`reduce(...)` derives `quantity_view` from `response_view`. Neither
changes the Plan or the original `response_view`.

The diagonal root is the loaded diagonal-root quantity of this selected
Direct reduction: the eliminated complement of the retained coordinate
has zero external current while the uncompensated Port loads remain. It
is neither an S-parameter dip nor the uncoupled LC formula.
`root_hint=6.0 * u.GHz` is an approximate branch locator, not an answer,
target, or search window. Then state the request and evaluate it.

In [ ]:
from scnsim import DiagonalRootSpec

root_spec = DiagonalRootSpec(
    coordinate=resonator_node,
    root_hint=6.0 * u.GHz,
)
root = run.evaluate(quantity_view, root_spec)

Read back both the explanation and the typed result independently.

In [ ]:
from IPython.display import display

run.explain(quantity_view, root_spec).show()
display(root.frequency)
display(root.linewidth)
display(root.slope)
root.show()

`resonator_node` is a root named-bus `ElectricNodeRef`. Internal LC
leaves are not exposed. Retaining it selects a boundary without changing
the Plan. The returned frequency locates the selected root, linewidth
describes its loaded width, and slope describes the local root behavior;
none is a new computation or a guarantee about a different observable.

[Previous](02_solve_direct.qmd) · [Next](04_define_parameters.qmd)